## Neural Networks - Deep Learning
**Theodora Tzina - AEM: 10715**
### Exercise 2
#### ***Part D: kNN - NCC - MLP: Comparison***

In [11]:
import time
import numpy as np
import dataProccessing as dp
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from keras import layers, models
from keras.utils import to_categorical

**CIFAR-10 dataset preparation**

Data augmentation was skipped due to computational cost:

In [3]:
# 1. Load CIFAR-10 dataset
x_train_raw, y_train_raw, x_test_raw, y_test_raw = dp.load_cifar10()

# 2. Convert CIFAR-10 labels to binary (Animals vs Vehicles)
y_train_bin, y_test_bin = dp.convert_labels(y_train_raw, y_test_raw)

# 3. Split training data into training and validation sets
x_train_split, y_train_split, x_val_split, y_val_split = dp.split_train_validation(x_train_raw, y_train_bin, val_size=0.4)

# 4. Balance CIFAR-10 dataset by undersampling majority class
x_train_bal, y_train_bal = dp.balance_dataset(x_train_split, y_train_split)

# 5. Extract HOG features from images
x_train_hog = dp.extract_hog_features(x_train_bal)
x_val_hog = dp.extract_hog_features(x_val_split)
x_test_hog = dp.extract_hog_features(x_test_raw)

# 6. Extract Color Histogram features from images
x_train_color = dp.extract_color_features(x_train_bal)
x_val_color = dp.extract_color_features(x_val_split)
x_test_color = dp.extract_color_features(x_test_raw)

# 7. Combine HOG and Color Histogram features
x_train_combined = np.hstack((x_train_hog, x_train_color))
x_val_combined = np.hstack((x_val_hog, x_val_color))
x_test_combined = np.hstack((x_test_hog, x_test_color))

print(f"---> Final feature count - Train: {x_train_combined.shape}, Val: {x_val_combined.shape}, Test: {x_test_combined.shape}")

# 8. Scale features to have zero mean and unit variance
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_combined)
x_val = scaler.transform(x_val_combined)
x_test = scaler.transform(x_test_combined)

# 9. Rename variables for clarity
y_train = y_train_bal
y_val = y_val_split
y_test = y_test_bin

Loading CIFAR-10 dataset
Original training data shape: (50000, 32, 32, 3)
Original test data shape: (10000, 32, 32, 3)
Flatten training labels shape: (50000,)
Flatten test labels shape: (10000,)

Converting CIFAR-10 labels to binary (Animals vs Vehicles)
CIFAR-10 binary super-classes:
0: Animals (bird, cat, deer, dog, frog, horse)
1: Vehicles (airplane, automobile, ship, truck)
Sample of converted training labels: [0 1 1 0 1 1 0 0 1 0]
Sample of converted test labels: [0 1 1 1 0 0 1 0 0 1]

Splitting training set into training (60.0%) and validation (40.0%)
New training data shape: (30000, 32, 32, 3)
Validation data shape: (20000, 32, 32, 3)

Balancing CIFAR-10 dataset by undersampling majority class
Class 0 samples before balancing: 18000
Class 1 samples before balancing: 12000
Balanced dataset size: 24000 samples
Class 0 samples: 12000
Class 1 samples: 12000

Extracting HOG features from images
HOG features shape: (24000, 324)

Extracting HOG features from images
HOG features shape: 

##### ***k-Nearest Neighbor***
Applying kNN (k=1, k=3) on dataset for comparison with SVM:

In [12]:
# Train and evaluate K-Nearest Neighbors classifier
for k in [1, 3]:

    print(f"K-Nearest Neighbors Classifier for k={k}")
    print("="*70)
        
    # Create and train the classifier
    knn = KNeighborsClassifier(n_neighbors=k)

    start_time = time.time()
    knn.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")
        
    # Predict on test set
    knn_accuracy = knn.score(x_test, y_test)
    print(f"Test set accuracy: {knn_accuracy:.2%}")
    print("="*70 + "\n")

K-Nearest Neighbors Classifier for k=1
Training time: 0.73 seconds
Test set accuracy: 84.74%

K-Nearest Neighbors Classifier for k=3
Training time: 0.01 seconds
Test set accuracy: 87.38%



##### ***Nearest Class Centroid***
Applying NCC on dataset for comparison with SVM:

In [13]:
# Train and evaluate Nearest Class Centroid classifier
print("Nearest Class Centroid Classifier")
print("="*70)

# Create and train the classifier
ncc = NearestCentroid()

start_time = time.time()
ncc.fit(x_train, y_train)
training_time = time.time() - start_time
print(f"Training time: {training_time:.2f} seconds")

# Predict on test set
ncc_accuracy = ncc.score(x_test, y_test)
print(f"Test set accuracy: {ncc_accuracy:.2%}")
print("="*70)

Nearest Class Centroid Classifier
Training time: 0.25 seconds
Test set accuracy: 81.76%


##### ***Multi-Layer Perceptron***
Applying MLP (1 hidden layer with Hinge loss) on dataset for comparison with SVM:

In [15]:
# Train and evaluate Multi-Layer Perceptron (MLP) with Hinge Loss
print("MLP with 1 hidden layer and Hinge Loss")
print("="*70 + "\n")

# Convert labels to one-hot encoding for MLP
num_classes = 2
y_train_onehot = to_categorical(y_train, num_classes)
y_val_onehot = to_categorical(y_val, num_classes)
y_test_onehot = to_categorical(y_test, num_classes)

# Define the MLP model
model = models.Sequential([
    # Input Layer: Matches your HOG features (324 features)
    layers.Input(shape=(x_train.shape[1],)),
    
    # Hidden Layer
    layers.Dense(100, activation='relu'), 

    # Output Layer
    layers.Dense(num_classes, activation='linear') 
])

# Compile the model with Hinge Loss
model.compile(
    optimizer='adam',
    loss='categorical_hinge',
    metrics=['accuracy']
)

# Train the model
start_time = time.time()
model.fit(x_train, y_train_onehot, epochs=20, batch_size=32, verbose=1,
          validation_data=(x_val, y_val_onehot))
training_time = time.time() - start_time
print(f"\nTraining time: {training_time:.2f} seconds")

# Evaluate the model on the test set
mlp_loss, mlp_accuracy = model.evaluate(x_test, y_test_onehot, verbose=0)
print(f"Test set accuracy: {mlp_accuracy:.2%}")
print("\n" + "="*70)

MLP with 1 hidden layer and Hinge Loss

Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8781 - loss: 0.3291 - val_accuracy: 0.8966 - val_loss: 0.2673
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9147 - loss: 0.2173 - val_accuracy: 0.9007 - val_loss: 0.2573
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9275 - loss: 0.1882 - val_accuracy: 0.9020 - val_loss: 0.2476
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9368 - loss: 0.1671 - val_accuracy: 0.9035 - val_loss: 0.2524
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9444 - loss: 0.1484 - val_accuracy: 0.9061 - val_loss: 0.2575
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9516 - loss: 0.1303 - val_accuracy: 0.9007 - val_loss: 0.2750
Epoch 7/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9575 - loss: 0.1157 - val_accuracy: 0.8999 - val_loss: 0.2936
Epoch 8/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 